In [1]:
# --- Day 16: Advanced Grover (Finding 101 and 110) ---
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit.library import GroverOperator, MCMT, ZGate

# 1. Define the Oracle (Marking 101 and 110)
# N = 8 (states 0 to 7). We want to mark 5 (101) and 6 (110).
oracle_qc = QuantumCircuit(3)

# Method: Phase Flip 101
# To target 101, we flip the middle bit (0) to make it 111, apply Controlled-Z, then flip back.
oracle_qc.x(1)
oracle_qc.cp(np.pi, 0, 2) # Controlled-Z between outer bits, controlled by middle? 
# Actually, let's use the Triple-Controlled Z (CCZ) logic standard.
# Simplified: We use MCMT (Multi-Controlled Multi-Target) gate for clean logic.

# A better way for learning: Use a Diagonal Phase Matrix
# This is how pro algorithms often insert oracles.
# List of phases: 1 means "Keep", -1 means "Mark"
# Index: 0, 1, 2, 3, 4, 5(101), 6(110), 7
diagonal_phases = [1, 1, 1, 1, 1, -1, -1, 1]
oracle_op = Operator.from_diagonal(diagonal_phases)

# 2. Build the Diffuser (Inversion about Mean)
def diffuser(n):
    qc = QuantumCircuit(n)
    qc.h(range(n))
    qc.x(range(n))
    
    # Apply Multi-Controlled Z (MCMT)
    # This flips the phase of |111>
    qc.compose(MCMT(ZGate(), n-1, 1), inplace=True)
    
    qc.x(range(n))
    qc.h(range(n))
    return qc

# 3. Construct the Grover Circuit
n = 3
qc = QuantumCircuit(n)

# A. Initialization
qc.h(range(n))

qc.barrier()

# B. Apply Grover Iteration (Oracle + Diffuser)
# We calculated that 1 iteration is optimal for 2 solutions in 8 items.
qc.unitary(oracle_op, range(n), label='Oracle (101 & 110)')
qc.compose(diffuser(n), inplace=True)

qc.barrier()

# C. Measure
qc.measure_all()

# 4. Draw
print("Grover Circuit (3 Qubits, 2 Solutions):")
display(qc.draw(output='mpl'))

# 5. Simulate
sim = AerSimulator()
qc_t = transpile(qc, sim)
counts = sim.run(qc_t, shots=1000).result().get_counts()

# 6. Analyze
print("\nResults (Look for 101 and 110):")
print(counts)
display(plot_histogram(counts))

AttributeError: type object 'Operator' has no attribute 'from_diagonal'